In [ ]:
import os
import pickle
import datetime
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tools import Evaluation_Tools

def preprocess_text(text):
    return text
    
target_similarity=defaultdict(list)
clause_weight_threshold = 10
number_of_examples = 1
clause_drop_p = 0.0
factor = 20
clauses = int(factor*20/(1.0 - clause_drop_p))
T = factor*40
s = 5.0
accumulation = 25
epochs = 300

et = Evaluation_Tools()
ds_path = files_start_name + '.csv'
pair_list = et.get_dataset_pairs(ds_path)

folder_path = 'datasets'
dir_count = 0
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        dir_count += 1
dataset_progress_bar = tqdm(total=dir_count, desc="Running Datasets")

for folder_name in os.listdir(folder_path):
    if folder_name == 'wordsim353-sim':
        current_folder_path = os.path.join(folder_path, folder_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, folder_name)

            f_vectorizer_X = open("vectorizer_X.pickle", "rb")
            vectorizer_X = pickle.load(f_vectorizer_X)
            f_vectorizer_X.close()
            
            with open(files_start_name + '_word1.pkl', 'rb') as f:
                word1 = pickle.load(f)
            with open(files_start_name + '_word2.pkl', 'rb') as f:
                word2 = pickle.load(f)
            word_total= list(set(word1 + word2))
            target_words=[]
            for i in word_total:
                if i in vectorizer_X.vocabulary_:
                    target_words.append(i)
            output= open(files_start_name + '_target.pkl', "wb")
            pickle.dump(target_words, output)
            output.close()

            with open(files_start_name + '_phase1.pkl', "rb") as file:
                output_active, feature_names, number_of_features, target_words_clauses = pickle.load(file)
                
            current_time = datetime.datetime.now()
            test_id = current_time.strftime("%Y%m%d%H%M%S")
            result_filename = f"{folder_name}_cross_phase2_{test_id}"
            test_start_name = os.path.join(current_folder_path, "tests")
            result_filepath = os.path.join(test_start_name , result_filename + '.txt')
            
            with open(result_filepath, 'w') as file, redirect_stdout(file):
                tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)
                total_training = 0
                print("Epochs: %d" % epochs)
                print("Target words: %d" % len(target_words))
                print("Accumulation: %d" % accumulation)
                print("Examples: %d" % number_of_examples)
                print("No of features: %d" % number_of_features)
                print("Clauses: %d\n" % clauses)
                
                epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                for e in range(epochs):
                    print("\nEpoch #: %d" % e)
                    start_training = time()
                    tm.clauses_fit(
                        number_of_examples = number_of_examples,
                        number_of_features = number_of_features,
                        target_words_clauses = target_words_clauses,
                        negative_weight_clause = True,  
                        cross_accumlation = True,
                        weight_insertion = False,
                        print_python = False,
                        print_c = False
                        )
                    stop_training = time()
                    epoch_time = stop_training - start_training
                    seconds = epoch_time
                    hours = seconds // 3600
                    minutes = (seconds % 3600) // 60
                    seconds = seconds % 60
                    print(f"Epoch training duration: {hours} hours, {minutes} minutes, {seconds} seconds")
                    total_training = total_training + epoch_time

                    profile = np.empty((len(target_words), clauses))
                    for i in range(len(target_words)):
                        weights = tm.get_weights(i)
                        profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                    similarity = cosine_similarity(profile)
                    for i in range(len(target_words)):
                        sorted_index = np.argsort(-1*similarity[i,:])
                        for j in range(1, len(target_words)):
                            target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                    et.evaluate(target_similarity,pair_list)
                    
                    epochs_progress_bar.update(1)
                epochs_progress_bar.close()

                print("\n=====================================\nClauses\n=====================================")
                for j in range(clauses):
                    print("Clause #%-2d " % (j), end=' ')
                    for tw in range(len(target_words)):
                        print("%s:W%-5d " % (target_words[tw], tm.get_weight(tw, j)), end='| ')
                    l = [] 
                    number_of_literals = 0 
                    for k in range(tm.clause_bank.number_of_literals):
                        if tm.get_ta_action(j, k) == 1:
                            number_of_literals = number_of_literals + 1
                            if k < tm.clause_bank.number_of_features:
                                l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                            else:
                                l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                    print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                    try:
                        print(" - ".join(l))
                    except UnicodeEncodeError:
                        print(" exception ")
                profile = np.empty((len(target_words), clauses))
                for i in range(len(target_words)):
                    weights = tm.get_weights(i)
                    profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                
                # output= open(files_start_name + '_knowledge_weights.pkl', "wb")
                # pickle.dump(profile, output)
                # output.close()
                
                print("\n=====================================\nWord Similarity\n=====================================")
                similarity = cosine_similarity(profile)
                max_word_length = len(max(target_words, key=len))
                list_of_words = []
                target_words_with_min_max = []
                for i in range(len(target_words)):
                    row_of_similarity = []
                    sorted_index = np.argsort(-1*similarity[i,:])
                    min_similarity = 1.0
                    max_similarity = 0.0
                    word_similarity = []
                    for j in range(1, len(target_words)):
                        target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                        row_of_similarity.append(target_words[sorted_index[j]])
                        word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
                        if(min_similarity > similarity[i,sorted_index[j]]):
                            min_similarity = similarity[i,sorted_index[j]]
                        if(max_similarity < similarity[i,sorted_index[j]]):
                            max_similarity = similarity[i,sorted_index[j]]
                
                    output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
                    print(output_line, end='     ==> ')
                    print(word_similarity)
                    list_of_words.append(row_of_similarity)
                    target_words_with_min_max.append(output_line)

                # output= open(files_start_name + '_knowledge_profile_dict.pkl', "wb")
                # pickle.dump(target_similarity, output)
                # output.close()

                seconds = total_training
                hours = seconds // 3600
                minutes = (seconds % 3600) // 60
                seconds = seconds % 60
                print(f"Training duration: {hours} hours, {minutes} minutes, {seconds} seconds")
    dataset_progress_bar.update(1)
dataset_progress_bar.close()

In [ ]:
import os
import pickle
import datetime
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tools import Evaluation_Tools

def preprocess_text(text):
    return text

target_similarity=defaultdict(list)
clause_weight_threshold = 0
number_of_examples = 100
clause_drop_p = 0.0
factor = 20
clauses = int(factor*20/(1.0 - clause_drop_p))
T = factor*40
s = 5.0
accumulation = 50
epochs = 10
et = Evaluation_Tools()

vectorizer_X = et.read_pickle_data("vectorizer_X.pickle")
folder_path = 'datasets'
dir_count = 0
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        dir_count += 1
dataset_progress_bar = tqdm(total=dir_count, desc="Running Datasets")
for folder_name in os.listdir(folder_path):
    if folder_name == 'wordsim353-sim':
        current_folder_path = os.path.join(folder_path, folder_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, folder_name)
            
            ds_path = files_start_name + '.csv'
            pair_list = et.get_dataset_pairs(ds_path)
            target_words = et.read_pickle_data(files_start_name + '_target.pkl')
            print("Target words: %d" % len(target_words))
            output_active, feature_names, number_of_features, target_words_clauses = et.read_pickle_data(files_start_name + '_phase1.pkl')
            target_words_pairs=[]
            for (pair,score) in pair_list:
                if pair[0] in target_words and pair[1] in target_words and pair[0] in vectorizer_X.vocabulary_ and pair[1] in vectorizer_X.vocabulary_:
                    target1_id = vectorizer_X.vocabulary_[pair[0]]
                    target2_id = vectorizer_X.vocabulary_[pair[1]]
                    target1_index = np.where(output_active == target1_id)[0]
                    target2_index = np.where(output_active == target2_id)[0]
                    target_words_pairs.append([target1_index[0],target2_index[0]])
            print("Target words pairs: %d" % len(target_words_pairs))

            current_time = datetime.datetime.now()
            test_id = current_time.strftime("%Y%m%d%H%M%S")
            result_filename = f"{folder_name}_phase2_{test_id}"
            test_start_name = os.path.join(current_folder_path, "tests")
            result_filepath = os.path.join(test_start_name , result_filename + '.txt')

            with open(result_filepath, 'w') as file, redirect_stdout(file):
                tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)
                total_training = 0
                print("Epochs: %d" % epochs)
                print("Target words: %d" % len(target_words))
                print("Pairs: %d" % len(pair_list))
                print("Accumulation: %d" % accumulation)
                print("Examples: %d" % number_of_examples)
                print("No of features: %d" % number_of_features)
                print("Clauses: %d\n" % clauses)
                
                epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                for e in range(epochs):
                    print("\nEpoch #: %d" % e)
                    start_training = time()
                    tm.clauses_pairs_fit(
                        number_of_examples = number_of_examples,
                        number_of_features = number_of_features,
                        target_words_clauses = target_words_clauses,
                        target_words_pairs = target_words_pairs,
                        negative_weight_clause = False,
                        print_python = False,
                        print_c = False
                        )
                    stop_training = time()
                    epoch_time = stop_training - start_training
                    seconds = epoch_time
                    hours = seconds // 3600
                    minutes = (seconds % 3600) // 60
                    seconds = seconds % 60
                    print(f"Epoch training duration: {hours} hours, {minutes} minutes, {seconds} seconds")
                    total_training = total_training + epoch_time

                    profile = np.empty((len(target_words), clauses))
                    for i in range(len(target_words)):
                        weights = tm.get_weights(i)
                        profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                    similarity = cosine_similarity(profile)
                    for i in range(len(target_words)):
                        sorted_index = np.argsort(-1*similarity[i,:])
                        for j in range(1, len(target_words)):
                            target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                    et.evaluate(target_similarity,pair_list)

                    epochs_progress_bar.update(1)
                epochs_progress_bar.close()

                print("\n=====================================\nClauses\n=====================================")
                for j in range(clauses):
                    print("Clause #%-2d " % (j), end=' ')
                    for tw in range(len(target_words)):
                        print("%s:W%-5d " % (target_words[tw], tm.get_weight(tw, j)), end='| ')
                    l = [] 
                    number_of_literals = 0 
                    for k in range(tm.clause_bank.number_of_literals):
                        if tm.get_ta_action(j, k) == 1:
                            number_of_literals = number_of_literals + 1
                            if k < tm.clause_bank.number_of_features:
                                l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                            else:
                                l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                    print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                    try:
                        print(" - ".join(l))
                    except UnicodeEncodeError:
                        print(" exception ")
                profile = np.empty((len(target_words), clauses))
                for i in range(len(target_words)):
                    weights = tm.get_weights(i)
                    profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                
                # output= open(files_start_name + '_knowledge_weights.pkl', "wb")
                # pickle.dump(profile, output)
                # output.close()
                
                print("\n=====================================\nWord Similarity\n=====================================")
                similarity = cosine_similarity(profile)
                max_word_length = len(max(target_words, key=len))
                list_of_words = []
                target_words_with_min_max = []
                for i in range(len(target_words)):
                    row_of_similarity = []
                    sorted_index = np.argsort(-1*similarity[i,:])
                    min_similarity = 1.0
                    max_similarity = 0.0
                    word_similarity = []
                    for j in range(1, len(target_words)):
                        target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                        row_of_similarity.append(target_words[sorted_index[j]])
                        word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
                        if(min_similarity > similarity[i,sorted_index[j]]):
                            min_similarity = similarity[i,sorted_index[j]]
                        if(max_similarity < similarity[i,sorted_index[j]]):
                            max_similarity = similarity[i,sorted_index[j]]
                
                    output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
                    print(output_line, end='     ==> ')
                    print(word_similarity)
                    list_of_words.append(row_of_similarity)
                    target_words_with_min_max.append(output_line)

                # output= open(files_start_name + '_knowledge_profile_dict.pkl', "wb")
                # pickle.dump(target_similarity, output)
                # output.close()

                seconds = total_training
                hours = seconds // 3600
                minutes = (seconds % 3600) // 60
                seconds = seconds % 60
                print(f"Training duration: {hours} hours, {minutes} minutes, {seconds} seconds")
    dataset_progress_bar.update(1)
dataset_progress_bar.close()